# Assignment 3: Queue-Aware Wireless Scheduling as an MDP

Author: *Giorgos Vassalos 2022030052*

#Task 1: Generic MDP formulation

***Define precisely the state space, action space and reward function***



1.   State Space (all the possible states of the scenario): To decide which user to serve the base station needs to know the amount of packets in each user's channel and state each user's channel is in so this translates into
$$s = (q_1, \dots, q_K, c_1, \dots, c_K)$$
where $q_i = (0,…,C)$ the amount of packets in the user's channel and $c_i = \{B,G\}$

2.   Action Space (the possible choices of the BS):  
$$A = (1,…,K)

3.  Reward function:
$$R_t = (1-λ) * T_i - λ * D_t$$
where λ is a parameter that tells the agent what to prioritize good throughput or full queues, $T_i$ is the amount of packets sent by user i and finally $D_t$ are the number of packets dropped during slot t.
---
***Derive the transition probabilities in generic form. You may present them either as formulas
or as a complete case distinction***
The transition probabilities are the mathematical odds of landing in any specific new state given the previous state and the action taken. To find the formula it is simple probabilities, more specifically we can calculate it by multiplying all the odds of each individual state space piece of happening:
$$Pr(s' \mid s, a) = \prod_{i=1}^K Pr(q_i^{new} \mid q_i') *Pr(c_i^{new} \mid c_i)$$


---
***For both the reward and the transition function, provide 2–3 illustrative examples using
concrete state-action pairs.***


# Task 2: Mathematical analysis of policy regimes


---


Our policy is "Always serve the user with the best current channel (regardless of queue sizes)" that means that λ = 0 since our reward is not reduced if we drop packets because we just choose the good channel each time. We also put γ=0 since our algorithm performs best if it is myopic due to the maximization of immediate reward and the lack of punishment from that short sightedness. Finally the last parameter is $p_i$ which is set as 1 to maximize transmitted packets (more packets in queue = more transmitted packets).


---
*Proof*

Let's analyze each parameter.


1.   λ=0: This parameter is required since the reward function penalizes for the loss of packets, however by putting λ=0 we make the final reward function to look like:
$$R_t = T_i(t)$$
which is optimal for what we want which is choosing the best performing channel each time and removing any care about dropping packets
2.   γ=0: This means that our algorithm needs to be myopic and care only about the highest immediate reward. This is true due to the fact that if we implement the agent to have future proofing then the policy of always picking the channel with the best throughput would be obsolete. To make our agent think that choosing the Best channel is optimal we mathematically remove his ability to have future sight.

3.   $p_i =1$: for the policy we are given to be the best we need the good channel to at least throughput as much as the bad channel which means that the amount of packets transmitted by choosing the good channel has to be greater or equal to the amount of packets transmitted if we chose the bad channel. So by putting the arrival probabilities of all users equal to 1 we achieve each user having at least one packet at the end of every slot.




#Task 3: Baselines, Policy Evaluation, and simulation

For the tiebreaker I will check the other policy and choose the one that would be chosen if the other policy is active for example if both channels are bad in the best channel policy I will choose the one with the most packets in the queue, contrary on the longest queue policy I will choose the channel that is good. If there is a tie again I will just choose user 1.

Initializing the parameters


In [1]:
import numpy as np
import itertools

#Parameters
K=2
C=4
N=2
gamma = 0.95

seed = 52
np.random.seed(seed)

p1 = np.random.uniform(0.6,0.9)
p2 = np.random.uniform(0.6,0.9)

pbg1 = np.random.uniform(0.1,0.3)
pbg2 = np.random.uniform(0.1,0.3)

pgb1 = np.random.uniform(0.1,0.3)
pgb2 = np.random.uniform(0.1,0.3)

l =np.random.uniform(0.2,0.8)

print(p1,p2,l)

0.8469331022129376 0.6078353944709601 0.23233413175906625


Creating the state space

In [2]:
q_vals = list(range(C+1))
c_vals = ['B', 'G']
state_space = list(itertools.product(q_vals,q_vals,c_vals,c_vals))
state_to_idx = {s: i for i, s in enumerate(state_space)}
n_states = len(state_space)

init_state = (0,0,'B','B')
init_idx = state_to_idx[init_state]

print(state_space)
print(state_to_idx)
print(n_states)

[(0, 0, 'B', 'B'), (0, 0, 'B', 'G'), (0, 0, 'G', 'B'), (0, 0, 'G', 'G'), (0, 1, 'B', 'B'), (0, 1, 'B', 'G'), (0, 1, 'G', 'B'), (0, 1, 'G', 'G'), (0, 2, 'B', 'B'), (0, 2, 'B', 'G'), (0, 2, 'G', 'B'), (0, 2, 'G', 'G'), (0, 3, 'B', 'B'), (0, 3, 'B', 'G'), (0, 3, 'G', 'B'), (0, 3, 'G', 'G'), (0, 4, 'B', 'B'), (0, 4, 'B', 'G'), (0, 4, 'G', 'B'), (0, 4, 'G', 'G'), (1, 0, 'B', 'B'), (1, 0, 'B', 'G'), (1, 0, 'G', 'B'), (1, 0, 'G', 'G'), (1, 1, 'B', 'B'), (1, 1, 'B', 'G'), (1, 1, 'G', 'B'), (1, 1, 'G', 'G'), (1, 2, 'B', 'B'), (1, 2, 'B', 'G'), (1, 2, 'G', 'B'), (1, 2, 'G', 'G'), (1, 3, 'B', 'B'), (1, 3, 'B', 'G'), (1, 3, 'G', 'B'), (1, 3, 'G', 'G'), (1, 4, 'B', 'B'), (1, 4, 'B', 'G'), (1, 4, 'G', 'B'), (1, 4, 'G', 'G'), (2, 0, 'B', 'B'), (2, 0, 'B', 'G'), (2, 0, 'G', 'B'), (2, 0, 'G', 'G'), (2, 1, 'B', 'B'), (2, 1, 'B', 'G'), (2, 1, 'G', 'B'), (2, 1, 'G', 'G'), (2, 2, 'B', 'B'), (2, 2, 'B', 'G'), (2, 2, 'G', 'B'), (2, 2, 'G', 'G'), (2, 3, 'B', 'B'), (2, 3, 'B', 'G'), (2, 3, 'G', 'B'), (2, 3, 'G

Defining the policies for user choice with implemented tie breakers

In [3]:
def best_channel(state):
  q1,q2,c1,c2 = state
  if c1 == 'G' and c2=='B':
    return 0
  elif c1 == 'B' and c2 == 'G':
    return 1
  else:
    if q1>=q2:
      return 0
    else:
      return 1


def longest_queue(state):
  q1,q2,c1,c2 = state
  if q1>q2:
    return 0
  elif q2>q1:
    return 1
  else:
    if c1 == 'G' and c2=='B':
      return 0
    elif c1 == 'B' and c2 == 'G':
      return 1
    else:
      return 0




Finding the expected reward and the probability matrix of each state happening (THEORETICAL)

In [4]:
def policy_eval_theory(policy):
  P = np.zeros((n_states, n_states))
  R = np.zeros(n_states)

  for idx, state in enumerate(state_space):
    q1,q2,c1,c2 = state
    action = policy(state)

    if action ==0:
      if c1 =='B':
        max_t = 1
      else:
        max_t = N
      final_t = min(q1 , max_t)

      q1_after = q1 - final_t
      q2_after = q2
    elif action ==1:
      if c2 == 'B':
        max_t = 1
      else:
        max_t = N
      final_t = min(q2 , max_t)

      q2_after = q2 - final_t
      q1_after = q1



    expected_reward = 0

    for user1_addition,user2_addition in itertools.product([0,1], [0,1]):
      probability = (p1 if user1_addition else 1-p1)*(p2 if user2_addition else 1-p2)

      if (q1_after + user1_addition)>C:
        drop1 = 1
        q1_next = q1_after
      else:
        q1_next = q1_after + user1_addition
        drop1 =0

      if (q2_after + user2_addition) >C:
        drop2 = 1
        q2_next = q2_after
      else:
        q2_next = q2_after + user2_addition
        drop2 =0

      drop_t = drop1+drop2

      r_t = (1 - l) * final_t - l * drop_t
      expected_reward += probability * r_t

      for c1_next, c2_next in itertools.product(c_vals,c_vals):
        if c1 == 'B':
          p_c1 = pbg1 if c1_next == 'G' else (1 - pbg1)
        else:
          p_c1 = pgb1 if c1_next == 'B' else (1 - pgb1)

        if c2 == 'B':
          p_c2 = pbg2 if c2_next == 'G' else (1 - pbg2)
        else:
          p_c2 = pgb2 if c2_next == 'B' else (1 - pgb2)

        total_p = probability * p_c1 * p_c2
        next_state = (q1_next,q2_next,c1_next,c2_next)
        next_idx = state_to_idx[next_state]

        P[idx,next_idx] += total_p
    R[idx]= expected_reward
  V = np.linalg.solve(np.eye(n_states) - gamma * P , R)
  return V[init_idx]

Simulation

In [5]:
def simulate_mdp(policy, mc, T):
  returns = []

  for mc_run in range(mc):
    q1,q2,c1,c2 = 0,0,'B','B'
    mc_return =0

    for t in range(T):
      action = policy((q1,q2,c1,c2))
      if action ==0:
        if c1 =='B':
          max_t = 1
        else:
          max_t = N
        final_t = min(q1 , max_t)

        q1_after = q1 - final_t
        q2_after = q2
      elif action ==1:
        if c2 == 'B':
          max_t = 1
        else:
          max_t = N
        final_t = min(q2 , max_t)

        q2_after = q2 - final_t
        q1_after = q1

      a1 = np.random.rand() < p1
      a2 = np.random.rand() < p2

      if (q1_after + a1)>C:
        drop1 = 1
        q1 = q1_after
      else:
        q1 = q1_after + a1
        drop1 =0

      if (q2_after + a2) >C:
        drop2 = 1
        q2 = q2_after
      else:
        q2 = q2_after + a2
        drop2 =0

      drop_t = drop1+drop2

      r_t = (1 - l) * final_t - l * drop_t
      mc_return += (gamma**t) * r_t

      if c1 =='B':
        if np.random.rand() <  pbg1:
          c1 = 'G'
        else:
          c1 = 'B'
      else:
        if np.random.rand() < pgb1:
          c1 = 'B'
        else:
          c1 = 'G'

      if c2 == 'B':
        if np.random.rand() < pbg2:
          c2 = 'G'
        else:
          c2 = 'B'
      else:
        if np.random.rand() < pgb2:
          c2 = 'B'
        else:
          c2 = 'G'

    returns.append(mc_return)

  return np.mean(returns)

Main

In [6]:
print("--- Best Channel Policy ---")
v_exact_bc = policy_eval_theory(best_channel)
v_sim_bc = simulate_mdp(best_channel, 2500 , 200)
print(f"Theory (Exact): {v_exact_bc:.4f}")
print(f"Simulation:     {v_sim_bc:.4f}")

print("\n--- Longest Queue Policy ---")
v_exact_lq = policy_eval_theory(longest_queue)
v_sim_lq = simulate_mdp(longest_queue, 2500 , 200)
print(f"Theory (Exact): {v_exact_lq:.4f}")
print(f"Simulation:     {v_sim_lq:.4f}")

--- Best Channel Policy ---
Theory (Exact): 17.0885
Simulation:     17.0427

--- Longest Queue Policy ---
Theory (Exact): 19.0786
Simulation:     19.0678


We notice that theoretical and simulation values are close but have small discrepencies. Those are caused to the perfection of our theoretical model which has an equivilent runtime of infinity in our simulation model which just runs for 2500 monte carlo runs for 200 horizon.

#Task 4: Optimal policy with Value Iteration

For this task we are asked to recreate the theoretical part of the last task however instead of having the agent's choices hard coded we try to find what the optimal move is using value iteration.

In [7]:
def build_mdp_no_policy():
    P_a = np.zeros((2, n_states, n_states))
    R_a = np.zeros((2, n_states))

    for idx, state in enumerate(state_space):
        q1, q2, c1, c2 = state

        for action in [0, 1]:
            if action == 0:
                max_t = 1 if c1 == 'B' else N
                final_t = min(q1, max_t)
                q1_after = q1 - final_t
                q2_after = q2
            else:
                max_t = 1 if c2 == 'B' else N
                final_t = min(q2, max_t)
                q2_after = q2 - final_t
                q1_after = q1

            expected_reward = 0

            for user1_addition, user2_addition in itertools.product([0, 1], [0, 1]):
                probability = (p1 if user1_addition else 1-p1) * (p2 if user2_addition else 1-p2)

                drop1 = max(0, (q1_after + user1_addition) - C)
                q1_next = min(C, q1_after + user1_addition)

                drop2 = max(0, (q2_after + user2_addition) - C)
                q2_next = min(C, q2_after + user2_addition)

                drop_t = drop1 + drop2
                r_t = (1 - l) * final_t - l * drop_t
                expected_reward += probability * r_t

                for c1_next, c2_next in itertools.product(c_vals, c_vals):
                    if c1 == 'B':
                        p_c1 = pbg1 if c1_next == 'G' else (1 - pbg1)
                    else:
                        p_c1 = pgb1 if c1_next == 'B' else (1 - pgb1)

                    if c2 == 'B':
                        p_c2 = pbg2 if c2_next == 'G' else (1 - pbg2)
                    else:
                        p_c2 = pgb2 if c2_next == 'B' else (1 - pgb2)

                    total_p = probability * p_c1 * p_c2
                    next_state = (q1_next, q2_next, c1_next, c2_next)
                    next_idx = state_to_idx[next_state]

                    P_a[action, idx, next_idx] += total_p

            R_a[action, idx] = expected_reward

    return P_a, R_a


def value_iteration(P_a, R_a, gamma, theta=1e-6):
    V = np.zeros(n_states)
    optimal_policy = np.zeros(n_states, dtype=int)

    while True:
        delta = 0
        Q = np.zeros((2, n_states))

        # Calculate Q-function: Q(s,a) = R(s,a) + gamma * sum(P(s'|s,a) * V(s'))
        for a in [0, 1]:
            Q[a] = R_a[a] + gamma * np.dot(P_a[a], V)

        new_V = np.max(Q, axis=0)
        #Calculating when the gap is small and stops the loop
        delta = np.max(np.abs(V - new_V))
        V = new_V

        if delta < theta:
            optimal_policy = np.argmax(Q, axis=0)
            break

    return V, optimal_policy

We notice a difference between the optimal policy that we produced using value iteration with the same gamma and both policies we applied in task 3. We see a smaller difference to the best channel policy than the longest queue policy. However both differ because of these reasons: The best channel policy has the flaw of allowing packets to drop as well as many times choosing a channel with 0 packets in it just because it is the best. The difference is small because our randomly generated l is a small number which means out reward cares only a little about dropped packets. Now about the longest queue policy it is much worse than both other because of its priority to be a safe system to send packets, avoid packet loss, instead of prioritizing throughput and in an environment that cares little about packet safety and cares more about the number of packets sent we get a smaller reward.

Moving on I will analyze the difference between the value iteration expected returns of gamma = 0.95 and gamma = 0.0. Gamma is a variable that shows how much in the reward calculation the future state takes part. So when we set it as 0 the agent only looks at the reward from the first step and since our first step has 0 packets on both queues it can't possibly produce reward. Now when gamma is 0.95 the agent looks ahead and calculates future reward. Of course its more complex but by making sacrifices now for the future it can get the highest expected reward of them all.

Calculating (i) reward per round, (ii) throughput per round, (iii) dropped packets per round.

In [8]:

P_a, R_a = build_mdp_no_policy()
V_95, pi_95 = value_iteration(P_a, R_a, gamma=0.95)

V_0, pi_0 = value_iteration(P_a, R_a, gamma=0.0)

def optimal_95_policy(state):
    return pi_95[state_to_idx[state]]

def optimal_0_policy(state):
    return pi_0[state_to_idx[state]]

def simulate_metrics(policy, mc=1000, T=200):
    total_reward = 0
    total_throughput = 0
    total_drops = 0
    total_steps = mc * T

    for _ in range(mc):
        q1, q2, c1, c2 = 0, 0, 'B', 'B'

        for t in range(T):
            action = policy((q1, q2, c1, c2))

            # 1. Transmission
            if action == 0:
                max_t = 1 if c1 == 'B' else N
                final_t = min(q1, max_t)
                q1_after = q1 - final_t
                q2_after = q2
            elif action == 1:
                max_t = 1 if c2 == 'B' else N
                final_t = min(q2, max_t)
                q2_after = q2 - final_t
                q1_after = q1

            a1 = np.random.rand() < p1
            a2 = np.random.rand() < p2

            drop1 = max(0, (q1_after + a1) - C)
            q1 = min(C, q1_after + a1)

            drop2 = max(0, (q2_after + a2) - C)
            q2 = min(C, q2_after + a2)

            drop_t = drop1 + drop2
            r_t = (1 - l) * final_t - l * drop_t

            # --- TRACKING RAW METRICS ---
            total_reward += r_t
            total_throughput += final_t
            total_drops += drop_t

            # 3. Channel Transitions
            if c1 == 'B':
                c1 = 'G' if np.random.rand() < pbg1 else 'B'
            else:
                c1 = 'B' if np.random.rand() < pgb1 else 'G'

            if c2 == 'B':
                c2 = 'G' if np.random.rand() < pbg2 else 'B'
            else:
                c2 = 'B' if np.random.rand() < pgb2 else 'G'

    # Return the average per round (step)
    return (total_reward / total_steps), (total_throughput / total_steps), (total_drops / total_steps)

policies_to_test = {
    "Best Channel Baseline": best_channel,
    "Longest Queue Baseline": longest_queue,
    "Optimal (Gamma = 0)": optimal_0_policy,
    "Optimal (Gamma = 0.95)": optimal_95_policy
}

print(f"{'Policy':<25} | {'Reward/Round':<15} | {'Throughput/Round':<18} | {'Drops/Round'}")
print("-" * 75)

for name, pol in policies_to_test.items():
    r, t, d = simulate_metrics(pol)
    print(f"{name:<25} | {r:<15.4f} | {t:<18.4f} | {d:.4f}")

Policy                    | Reward/Round    | Throughput/Round   | Drops/Round
---------------------------------------------------------------------------
Best Channel Baseline     | 0.9376          | 1.2701             | 0.1613
Longest Queue Baseline    | 1.0550          | 1.3885             | 0.0471
Optimal (Gamma = 0)       | 1.0532          | 1.3863             | 0.0476
Optimal (Gamma = 0.95)    | 1.0665          | 1.4003             | 0.0367
